# Third-order shear deformation theory with a laminate offset: connections and reference surface

With Reddy's third-order shear deformation theory (TSDT) and a laminate offset $d$, the mid-surface of the laminate is
at $z = d$ from the reference surface $z = 0$ of the displacements, and the in-plane displacement is
$u(z) = u + z \phi_x - c_1 [(z - d)^3 + d^3] (\phi_x + w_{,x})$, $c_1 = 4/(3h^2)$, so that the transverse shear strain
vanishes at both faces $z = d \pm h/2$, see
[Third-order shear deformation theory with a laminate offset](../tsdt_offset.rst). The rotations of the normal at the
reference surface are

$$\theta_x = \phi_x - \frac{4 d^2}{h^2} (\phi_x + w_{,x}), \qquad \theta_y = \phi_y - \frac{4 d^2}{h^2} (\phi_y +
w_{,y}),$$

and the connections of [`MultiDomain`](../multidomain.rst) tie $u, v, w, \theta_x, \theta_y$ at $z = 0$, the surface
shared by the connected panels. This notebook checks, with the models of the test modules
`tests/multidomain/test_tsdt_offset_connections.py` and `tests/tests_shell/test_tsdt_offset.py`:

1. two panels with different offsets, connected exactly: the rotations at the reference surface are continuous across
   the connection, those at the mid-surfaces are not;
2. two panels with the same offset reproduce the single panel, with both connection methods;
3. a free homogeneous plate is the same structure for any reference surface: its frequencies do not depend on $d$.

**Reference.** J. N. Reddy, *Mechanics of Laminated Composite Plates and Shells: Theory and Analysis*, 2nd ed., CRC
Press, 2003, [doi:10.1201/b12409](https://doi.org/10.1201/b12409).

In [1]:
%matplotlib inline
import os
import sys

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from structsolve import solve

sys.path.insert(0, os.path.join('..', 'tests', 'multidomain'))
sys.path.insert(0, os.path.join('..', 'tests', 'tests_shell'))
from test_tsdt_offset_connections import H, _plate, _two_plates, _rotations
from test_tsdt_offset import _free_plate_freqs
from structsolve.sparseutils import remove_null_cols
from scipy.linalg import eigh


def table(headers, rows):
    lines = ['| ' + ' | '.join(headers) + ' |', '|' + '---|'*len(headers)]
    lines += ['| ' + ' | '.join(str(v) for v in row) + ' |' for row in rows]
    display(Markdown('\n'.join(lines)))


def frequencies(K, M, k):
    K, _ = remove_null_cols(K, silent=True)
    M, _ = remove_null_cols(M, silent=True)
    return np.sqrt(eigh(K.toarray(), M.toarray(), eigvals_only=True, subset_by_index=[0, k - 1]))

## 1. Continuity of the rotations at the reference surface

Two simply supported plates, $0.4 \times 1$ and $0.6 \times 1$, $h = 0.1$, cross-ply $[0/90/0/90]$ with $E_1/E_2 = 10$,
$10 \times 10$ terms, connected along $x = 0.4$ by ``'SSxcte'`` with the null-space method, under a uniform pressure of
$-10^4$ Pa. The first plate has no offset, $d_1 = 0$, the second $d_2 = 0.3h$, $h/2$ or $-h/2$. The rotations are
evaluated on both sides of the connection at seven points along $y$, from the fields $\phi_x$, $\phi_y$ and the
transverse shear strains $\gamma_{xz} = \phi_x + w_{,x}$, $\gamma_{yz} = \phi_y + w_{,y}$ of
[`Shell.strain`](../shell.rst). Jumps relative to the largest $|\theta_x|$.

In [2]:
ys = np.linspace(0.05, 0.95, 7)
rows = []
for d2 in (0.3*H, 0.5*H, -0.5*H):
    md, p1, p2 = _two_plates(0., d2, conn_method='null-space')
    p1.add_pressure_load(-1.e4)
    p2.add_pressure_load(-1.e4)
    c = md.expand(solve(md.reduce(md.calc_kC()), md.reduce(md.calc_fext()), silent=True))
    th1x, th1y, phi1x, _ = _rotations(p1, c, np.full_like(ys, p1.a), ys)
    th2x, th2y, phi2x, _ = _rotations(p2, c, np.zeros_like(ys), ys)
    scale = np.abs(th1x).max()
    rows.append(['%+.1f h' % (d2/H), '%.1e' % (np.abs(th1x - th2x).max()/scale),
                 '%.1e' % (np.abs(th1y - th2y).max()/scale), '%.1e' % (np.abs(phi1x - phi2x).max()/scale)])
table(['$d_2$', 'jump of $\\theta_x$', 'jump of $\\theta_y$', 'jump of $\\phi_x$'], rows)

| $d_2$ | jump of $\theta_x$ | jump of $\theta_y$ | jump of $\phi_x$ |
|---|---|---|---|
| +0.3 h | 1.1e-15 | 1.2e-15 | 9.0e-01 |
| +0.5 h | 1.3e-15 | 1.3e-15 | 1.7e+00 |
| -0.5 h | 1.4e-15 | 2.0e-15 | 1.7e+00 |

The connection ties the rotations at the common reference surface; the rotations at the mid-surfaces differ by the
transverse shear term $\frac{4 d^2}{h^2} \gamma_{xz}$ when $d_1 \ne d_2$.

## 2. Two panels with the same offset against the single panel

The same plates, both with $d = 0.3h$, $14 \times 14$ terms, against the single plate with $18 \times 18$ terms: first
six natural frequencies, with the penalty method (constants of `calc_kt_kr` multiplied by 1000) and with the null-space
method. Relative difference $\omega/\omega_{single} - 1$.

In [3]:
d = 0.3*H
single = _plate(1., 1., d, 'SSSS', m=18, n=18)
ref = frequencies(single.calc_kC(), single.calc_kM(), 6)
rows = [['single plate'] + ['%.4f' % w for w in ref]]
for method in ('penalty', 'null-space'):
    md, p1, p2 = _two_plates(d, d, conn_method=method, m=14)
    K, M = md.reduce(md.calc_kC()), md.reduce(md.calc_kM(silent=True))
    om = frequencies(K, M, 6)
    rows.append([method] + ['%.4f (%+.1e)' % (w, w/r - 1) for w, r in zip(om, ref)])
table(['model'] + ['$\\omega_%d$ (rad/s)' % (i + 1) for i in range(6)], rows)

| model | $\omega_1$ (rad/s) | $\omega_2$ (rad/s) | $\omega_3$ (rad/s) | $\omega_4$ (rad/s) | $\omega_5$ (rad/s) | $\omega_6$ (rad/s) |
|---|---|---|---|---|---|---|
| single plate | 762.9651 | 1788.0271 | 1791.7666 | 1894.5035 | 1896.1284 | 2652.5489 |
| penalty | 762.9335 (-4.1e-05) | 1788.0214 (-3.2e-06) | 1791.7529 (-7.6e-06) | 1894.4890 (-7.7e-06) | 1896.0799 (-2.6e-05) | 2652.5129 (-1.4e-05) |
| null-space | 762.9655 (+5.0e-07) | 1788.0289 (+1.0e-06) | 1791.7718 (+2.9e-06) | 1894.5036 (+4.2e-08) | 1896.1289 (+3.1e-07) | 2652.5500 (+4.1e-07) |

## 3. Free plate: independence of the reference surface

A free homogeneous aluminium plate, $0.6 \times 0.4$ m, $h = 20$ mm, $E = 70$ GPa, $\nu = 0.3$, $8 \times 7$ terms. With
free edges the Ritz spaces of the different offsets coincide, the structure is the same, and so must be the natural
frequencies (the six rigid-body modes are excluded).

In [4]:
f0 = _free_plate_freqs(0.)
rows = [['0', ] + ['%.4f' % f for f in f0]]
offsets = (0.01, -0.013, 0.025)
for offset in offsets:
    f = _free_plate_freqs(offset)
    rows.append(['%g m' % offset] + ['%.4f (%+.1e)' % (x, x/y - 1) for x, y in zip(f, f0)])
table(['offset $d$'] + ['$\\omega_%d$ (rad/s)' % (i + 1) for i in range(len(f0))], rows)

| offset $d$ | $\omega_1$ (rad/s) | $\omega_2$ (rad/s) | $\omega_3$ (rad/s) | $\omega_4$ (rad/s) | $\omega_5$ (rad/s) | $\omega_6$ (rad/s) |
|---|---|---|---|---|---|---|
| 0 | 1698.2078 | 1824.9875 | 3902.2306 | 4231.7889 | 4866.7483 | 5642.5925 |
| 0.01 m | 1698.2078 (+1.3e-12) | 1824.9875 (-4.8e-13) | 3902.2306 (+3.8e-13) | 4231.7889 (-1.7e-14) | 4866.7483 (-5.2e-14) | 5642.5925 (+2.7e-13) |
| -0.013 m | 1698.2078 (+3.7e-12) | 1824.9875 (+3.9e-12) | 3902.2306 (-7.0e-13) | 4231.7889 (-7.8e-13) | 4866.7483 (-4.5e-13) | 5642.5925 (-7.1e-14) |
| 0.025 m | 1698.2078 (-7.4e-12) | 1824.9875 (+7.0e-13) | 3902.2306 (-1.9e-13) | 4231.7889 (-1.5e-12) | 4866.7483 (-1.3e-12) | 5642.5925 (+2.4e-13) |

## Discussion

The rotations of the connected panels are continuous at their common reference surface to round-off, panels with the
same offset reproduce the single panel to the convergence of the approximation (within $10^{-4}$ here, with both
methods), and the frequencies of the free plate do not depend on the reference surface, to round-off.